This notebook demonstrates how to fine-tune a Persian GPT-2 model ([HooshvareLab/gpt2-fa](https://huggingface.co/HooshvareLab/gpt2-fa)) for conditional poetry generation ([HooshvareLab/gpt2-fa-poetry](https://huggingface.co/HooshvareLab/gpt2-fa-poetry)). HooshvareLab/gpt2-fa, a Persian-optimized version of GPT-2 developed by HooshvareLab.

The model learns to generate poetry in the style of specific Persian poets based on a prompt containing the poet's name.

In [6]:
!pip install -qU transformers
#!pip install -qU hazm

In [ ]:
'''
import hazm


normalizer = hazm.Normalizer(persian_numbers=False)

def normalize_input(text):
    text = normalizer.normalize(text)
    return text
'''

In [2]:
# Load dataset
!git clone https://github.com/aghasemi/ChronologicalPersianPoetryDataset.git poems

Cloning into 'poems'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 39 (delta 2), reused 0 (delta 0), pack-reused 33 (from 1)
Receiving objects: 100% (39/39), 34.27 MiB | 9.44 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (8/8), done.


In [7]:
import json
import os
import glob

import numpy as np
import pandas as pd

from tqdm import tqdm

In [8]:
df = pd.read_csv("/content/poems/poems.tsv", sep="\t")


In [9]:
# Depends on you
#df["poem"] = df["poem"].apply(lambda t: normalize_input(t))

df["text"] = df["poet"] + "<|startoftext|>" + df["poem"].apply(lambda t: t.replace("    ", "<sep>").replace("\t", "<sep>"))
df = df.dropna()
df = df.reset_index(drop=True)
df.head()

,poet,century,poem,text
0,رودکی,3,گر من این دوستی تو ببرم تا لب گور بزنم نعره...,رودکی<|startoftext|>گر من این دوستی تو ببرم تا...
1,رودکی,3,به حق نالم ز هجر دوست زارا سحرگاهان چو بر گ...,رودکی<|startoftext|>به حق نالم ز هجر دوست زارا...
2,رودکی,3,به نام نیک تو، خواجه، فریفته نشوم که نام نی...,رودکی<|startoftext|>به نام نیک تو، خواجه، فریف...
3,رودکی,3,دلا، تا کی همی جویی منی را؟ چه داری دوست هر...,رودکی<|startoftext|>دلا، تا کی همی جویی منی را...
4,رودکی,3,گرفت خواهم زلفین عنبرین ترا به بوسه نقش‌کنم...,رودکی<|startoftext|>گرفت خواهم زلفین عنبرین تر...


In [10]:
poets = list(df["poet"].unique())

In [13]:
idx = np.random.randint(0, len(df))
print(idx)
print(df.iloc[idx]["poet"])
print(df.iloc[idx]["poem"])
print(df.iloc[idx]["text"])

5540
سنایی
ای ز آب زندگانی آتشی افروخته    واندر او ایمان و کفر عاشقان را سوخته    ای تف عشقت به یک ساعت به چاه انداخته    هر چه در صد سال عقل ما ز جان اندوخته    ای کمالت کمزنان را صبرها پرداخته    وی جمالت مفلسان را کیسه‌ها بردوخته    گه به قهر از جزع مشکین تیغها افراخته    گه به لطف از لعل نوشین شمعها افروخته    هر چه در سی سال کرده خاتم مشکینت وام    آن نگین لعل نوشین در زمانی توخته    ما به جان بخریده عشق لایزالی را تو باز    لاابالی گفته و بر ما جهان بفروخته    ای ز آب روی خویش اندر دبیرستان عشق    تختهٔ عمر سنایی شسته از آموخته
سنایی<|startoftext|>ای ز آب زندگانی آتشی افروخته<sep>واندر او ایمان و کفر عاشقان را سوخته<sep>ای تف عشقت به یک ساعت به چاه انداخته<sep>هر چه در صد سال عقل ما ز جان اندوخته<sep>ای کمالت کمزنان را صبرها پرداخته<sep>وی جمالت مفلسان را کیسه‌ها بردوخته<sep>گه به قهر از جزع مشکین تیغها افراخته<sep>گه به لطف از لعل نوشین شمعها افروخته<sep>هر چه در سی سال کرده خاتم مشکینت وام<sep>آن نگین لعل نوشین در زمانی توخته<sep>ما به جان بخریده عشق لایزالی را تو باز<sep>لااب

In [14]:
import numpy as np

from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

from IPython import display

In [15]:
model_name_or_path = "HooshvareLab/gpt2-fa"

tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path,
    bos_token='<s>',
    eos_token='</s>',
    pad_token='<pad>',
    unk_token='<unk>'
)
tokenizer.add_special_tokens({
    "bos_token": '</s>',
    "eos_token": '</s>',
    "pad_token": '<pad>',
    "unk_token": '<unk>'
})

config = AutoConfig.from_pretrained(
    model_name_or_path,
    bos_token_id=tokenizer("<s>")["input_ids"][0],
    eos_token_id=tokenizer("</s>")["input_ids"][0],
    pad_token_id=tokenizer("<pad>")["input_ids"][0],
    unk_token_id=tokenizer("<unk>")["input_ids"][0],
)

tokenizer.save_pretrained("/content/gpt2/")
config.save_pretrained("/content/gpt2/")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [16]:
!wget "https://huggingface.co/HooshvareLab/gpt2-fa/resolve/main/pytorch_model.bin" -P /content/gpt2/
!wget "https://huggingface.co/HooshvareLab/gpt2-fa/resolve/main/tokenizer.json" -P /content/gpt2/

--2025-05-18 08:30:18--  https://huggingface.co/HooshvareLab/gpt2-fa/resolve/main/pytorch_model.bin
Resolving huggingface.co (huggingface.co)... 3.165.160.11, 3.165.160.12, 3.165.160.59, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.11|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/HooshvareLab/gpt2-fa/46b0b806c740a0f0a9f056f5574c5fa896166fe844945fd3c849bf34365e5060?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27pytorch_model.bin%3B+filename%3D%22pytorch_model.bin%22%3B&response-content-type=application%2Foctet-stream&Expires=1747560618&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NzU2MDYxOH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9Ib29zaHZhcmVMYWIvZ3B0Mi1mYS80NmIwYjgwNmM3NDBhMGYwYTlmMDU2ZjU1NzRjNWZhODk2MTY2ZmU4NDQ5NDVmZDNjODQ5YmYzNDM2NWU1MDYwP3Jlc3BvbnNlLWNvbnRlbnQtZGlzcG9zaXRpb249KiZyZXNwb25zZS1jb250ZW50LXR5cGU9KiJ9XX0_&Signature=KGb9OmaBA6igNKLsYccL

In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    "/content/gpt2",
    bos_token='<s>',
    eos_token='</s>',
    pad_token='<pad>'
)

print(tokenizer.encode("سلام بر شما"))
print(tokenizer.encode("<s>"))
print(tokenizer.encode("</s>"))
print(tokenizer.encode("<pad>"))
print(tokenizer.encode("<|startoftext|>"))
print(tokenizer.encode("<sep>"))

[8906, 327, 512]
[0]
[2]
[1]
[6]
[9]


In [21]:
texts = df["text"].values.tolist()

print(len(texts))

54524


In [22]:
df[df['poet']=='حافظ']

,poet,century,poem,text
24358,حافظ,8,جز نقش تو در نظر نیامد ما را جز کوی تو رهگذ...,حافظ<|startoftext|>جز نقش تو در نظر نیامد ما ر...
24359,حافظ,8,بر گیر شراب طرب‌انگیز و بیا پنهان ز رقیب سف...,حافظ<|startoftext|>بر گیر شراب طرب‌انگیز و بیا...
24360,حافظ,8,گفتم که لبت، گفت لبم آب حیات گفتم دهنت، گفت...,حافظ<|startoftext|>گفتم که لبت، گفت لبم آب حیا...
24361,حافظ,8,ماهی که قدش به سرو می‌ماند راست آیینه به دس...,حافظ<|startoftext|>ماهی که قدش به سرو می‌ماند ...
24362,حافظ,8,من باکمر تو در میان کردم دست پنداشتمش که در...,حافظ<|startoftext|>من باکمر تو در میان کردم دس...
...,...,...,...,...
27427,حافظ,8,به جد و جهد چو کاری نمی رود از پیش به کردگا...,حافظ<|startoftext|>به جد و جهد چو کاری نمی رود...
27428,حافظ,8,رهروان را عشق بس باشد دلیل آب چشم اندر رهش ...,حافظ<|startoftext|>رهروان را عشق بس باشد دلیل<...
27429,حافظ,8,روز عید است و من امروز در آن تدبیرم که دهم ...,حافظ<|startoftext|>روز عید است و من امروز در آ...
27430,حافظ,8,ای در چمن خوبی رویت چو گل خودرو چین شکن زلف...,حافظ<|startoftext|>ای در چمن خوبی رویت چو گل خ...


In [23]:
texts[24358]

'حافظ<|startoftext|>جز نقش تو در نظر نیامد ما را<sep>جز کوی تو رهگذر نیامد ما را<sep>خواب ارچه خوش آمد همه را در عهدت<sep>حقا که به چشم در نیامد ما را'

In [24]:
max_seq = 256

In [25]:
from torch.utils.data import Dataset
import torch

torch.manual_seed(42)

class MTGDataset(Dataset):
    def __init__(self, txt_list, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.data = []

        for txt in txt_list:
            encodings_dict = tokenizer(
                '<s>' + txt + '</s>',
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )

            self.data.append({
                "input_ids": encodings_dict["input_ids"].squeeze(0),  # Remove batch dim
                "attention_mask": encodings_dict["attention_mask"].squeeze(0),
                "labels": encodings_dict["input_ids"].squeeze(0)  # Labels are the same as input
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]  # Returns a dict (Trainer requires dict format)


In [26]:
from torch.utils.data import random_split

dataset = MTGDataset(texts, tokenizer, max_length=max_seq)

# Split into training and validation sets
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

f'There are {len(train_dataset)} samples for training, and {len(val_dataset)} samples for validation testing'

'There are 49071 samples for training, and 5453 samples for validation testing'

---

In [27]:
index = 1053
print(train_dataset[index]['input_ids'][:100])
print(tokenizer.decode(train_dataset[index]['input_ids'][:100]))

tensor([    0, 40719,     6,   476,  1061,  5634,   953,   991,   403,   312,
         1352,  9540,     9, 29930,   287,   421,  9268, 11042,  3645,   327,
         4141,   293,  7323,     9,  8101,   303,   798, 18085,   390,   326,
         2728, 18369,     9,  2076,   817,  1013,   484,  1733,  1801,   303,
          798, 18085,     9,   316, 24744,   571,   330, 39148,   546,   303,
         3590,  6306,   742,     9,   278,  5748,  8503,   395,   312,   451,
         2167,   440,  8768,   751,     9,  2076, 11392,   960,   318,  3645,
          326,  1323, 26680,  2511, 15987,     9,   458,  3735,   314,  4446,
        20927,   310,   285, 19581,  3645,  7935,     9,  2802,   773,  3982,
         8776,   457,   293, 14162,  7607,   486,   451,     9,   589,   747])
<s>حافظ<|startoftext|>افسر سلطان گل پیدا شد از طرف چمن<sep>مقدمش یا رب مبارک باد بر سرو و سمن<sep>خوش به جای خویشتن بود این نشست خسروی<sep>تا نشیند هر کسی اکنون به جای خویشتن<sep>خاتم جم را بشارت ده به حسن خاتمت<sep>کاس

#Training model using huggingface Trainer API

---



In [28]:
import random
import numpy as np
import torch
from transformers import AutoConfig, AutoModelForCausalLM

# Load configuration automatically
configuration = AutoConfig.from_pretrained('./gpt2', output_hidden_states=False)

# Load model with the config
model = AutoModelForCausalLM.from_pretrained("./gpt2", config=configuration)
model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda")
model.to(device)

# Optional: for reproducibility
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)


In [ ]:
import torch
import random
import time
import datetime
from transformers import Trainer, TrainingArguments

epochs = 3


# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",  # Evaluate at the end of each epoch
    save_strategy="steps",
    save_steps=2000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=epochs,
    logging_dir="./logs",
    logging_steps=2000,
    save_total_limit=3,           # Keep only last 3 checkpoints
    load_best_model_at_end=True,
    report_to="none",             # Avoid sending logs to external trackers
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    # compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Evaluate
trainer.evaluate()


<ipython-input-29-dd08751bc724>:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
2000,2.935400,2.728100


Step,Training Loss,Validation Loss
2000,2.935400,2.728100
4000,2.717300,2.615328
6000,2.612300,2.545464
8000,2.499100,2.504992


In [ ]:
import re


def generator(model, poet, max_length=128, num_return_sequences=3):
    model.eval()
    prompt = f"<s>{poet}<|startoftext|>"
    print(prompt)

    generated = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
    generated = generated.to(device)

    decoded_outputs = model.generate(
        generated,
        do_sample=True,
        top_k=50,
        max_length=max_length,
        top_p=0.95,
        num_return_sequences=num_return_sequences
    )


    outputs = []
    for i, output in enumerate(decoded_outputs):
        o = tokenizer.decode(output, skip_special_tokens=False)
        o = o.replace("<s>", "").replace("</s>", "")
        outputs.append(o)



    display.display(display.HTML("""
    <style>
    @import url("https://cdn.jsdelivr.net/gh/rastikerdar/vazir-font@v27.1.0/dist/font-face.css");

    table.xxx {
        margin-right: 15px;
        font-size: 14px;
        direction: rtl !important;
        width: 100%;
        display: flex;
    }
    table.xxx td {
        min-width: 300px !important;
        direction: rtl !important;
        text-align: right !important;
        font-family: "Vazir" !important;
    }
    </style>
    """.strip()))


    df = pd.DataFrame(outputs, columns=["generated"])
    df["generated"] = df["generated"].apply(lambda t: re.sub("\n+", "\n", t.replace("<sep>", "\n")))
    df["generated"] = df["generated"].apply(lambda t: "<p>" + t.replace("\n", "<br/>").replace("<|startoftext|>", "<br/>").strip() + "</p>")


    setup = {
        'border': 2,
        'show_dimensions': True,
        'escape': False,
        'justify': 'right',
        'classes': 'xxx'
    }
    display.display(display.HTML(df.to_html(**setup)))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = 4000  # after 4000 iteration
# Load the best trained model
model_path = "./results/checkpoint-4000"  # or any directory where the best model is saved
hf_model = AutoModelForCausalLM.from_pretrained(model_path)
hf_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hf_model.to(device)
hf_model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(42001, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=42001, bias=False)
)

In [ ]:
generator(hf_model, 'حافظ', num_return_sequences=2)

<s>حافظ<|startoftext|>


,generated
0,حافظای بی‌تو که در عالم افکنی چو حبابیا رب ای مرغزار و مرغزار و مرغزاراز پی خلق جهان را بشکافبر تو ز چه رو دست می‌بندی چو حبابدیوانه شو که نه من خود را از خود جدا می‌کنمدر ره تو چو بر تو هم بی‌اختیار می‌کنمای دل چو در کار تویی و چو تو نیز در کارتا به کی این راه به مقصد می‌کنی چو حباباین غزل غزل اگر آن روز به تو رسید دلبهر آن که به تو این جا نیز می‌کنی چو حباب
1,حافظبگذر از لب یار که جان من استمن دل و جان به من که جان من استبگذر ز خود آن دل که با دلم نشستدلم به من به جان که جان من استعرفی به زلف تو نتوان کرد دلگر نه در این کار که جان من استمن ز وصل تو جان به جان و جان می‌دهمکز دل و جان به جان جان من استمطلب به کام دل که در دست استخوشا که تا زنده‌ام از دل دوستتا به کی بر این خاکدان جان من است


In [ ]:
generator(hf_model, 'سهراب سپهری', num_return_sequences=2)

<s>سهراب سپهری<|startoftext|>


,generated
0,سهراب سپهریدر این دشت پرآشوب، بیقراری ها می کنند، و به تاریکی های خود میپرند. با این که این بیابان ها را به یکدیگر آورده است، اما اینجا، جاییست که از سادگی های دیگر لذت می برد. ، برای من هر شبی که می رفتم و به خواب رفته ام، و برای من بی خواب، و اگر می خوردم، به جایی، که تنها، مرا از برای خود می برند، و من بی خواب به تنهایی هستم. در این خانه اما، نمی توانم گذشت، و من بی خیال. اگر چه، اما، می دانم که چگونه در این مکان بیگانه،
1,سهراب سپهریدر آن پنجره که از تو بیرون نیستپرتو را در سر ما و بیرون نیستآه، که می سوزد بر من، ای خوش، که در دل شببا من ای خوش نشین، ای بی باک، ای خوش نشین،در این میان، گر چه غم، غم از دل، آیداز تو، می سوزد خون من، به امید دل، که در شب،گل می نوش، ای خوش می نوش، ای خوش خوش بیا، ای خوشگل می می بارد از دست و گر بر خاک می زنندبر سر ما، به هر گلی که


In [ ]:
generator(pretrained_model, 'حافظ', num_return_sequences=2)

<s>حافظ<|startoftext|>


,generated
0,حافظای دل مکن از غصه که او یار منستدل شد گرفتار تو و دلدار منستبگذار ای طبیب که بیمار منستدل نیست جز آن غمزه که بیمار منستگوئی که صبر گوشه‌ای از من بگیرکانجا که عشق بر سر بازار منستسرها به باد رفت و من از دست رفت دلیارب چه کار در سر دستار منستمشنو ز من که از می و معشوق مست استمستقبلانه عشق که بسیار منستبا آن که در دل و سینه دارم امید نیستاین در وصال بود و در یار
1,حافظیاد باد آنروز که از چشم تو ما را گله بودیمحال است که در صومعه با دلق تناسخ بودیمبا غیر خدا گفتم و ما را بطلبیدندمی‌خواست که این یار هم از خویش بود هم از خدا همگفتم نتوان گفت که از کوی خرابات درآمدگفت از نظرم غیر خدا هیچ مپرسیددل گفت بدارم به در صومعه با خود گفتمگفت از کرم شاه به خلوتگه شهنشاهحافظ ز برای سخن آمد همه را گفتتا آن که در میخانه دم از صدق زد و داداز دست و دلش توبه شکن بود و می و جامگفتم


In [ ]:
generator(pretrained_model, 'شهریار', num_return_sequences=2)

<s>شهریار<|startoftext|>


,generated
0,شهریارشب از من برد و غم از دل نبرد غمگسار ماشب از من برد و دل از جان نبرد غمگسار ماچو لاله از دل خونین فروچکید اشک و بازغزل آمد به گوش گل صدای سار ماز آه سینه سوز سینه سوز ما گذشتز دود سینه دم سرد روزگار مابه کام دل ز مژگان جگر خراش دمیدچو آه گرم از دل بی تاب شهریار ماسپهر ما به سرگشتگی و چشم اشکباربه سوی آسمان چارده رفت و چارماغبار غم به دامان کوه غم نشستغبار غم به دامان کوه غم نشست
1,شهریارهر که از جور تو بی بهره نه مرد سفرستیارب این دیده که گریان شده شبگیر درستگرنه تنها به تو ای تازه گل باغ حیاتگل سوری چرا بر رخ یاران به درستبا غم عشق تو یک عمر چه باشد گرمیمکاروان همه شب ناله زار و جگرستگر چه عمری به سر کوی تو می آیم بازچشم گریان ز پی همرهی باد سحرستگر چه در سینه من عشق تو بی سوز بودآه من باد کز او ناله نی هم جگرستدر گذر از دو سه روزی که به بازار وجودهر دو بازار وجود از


In [ ]:
 generator(pretrained_model, 'سهراب سپهری', num_return_sequences=2)

<s>سهراب سپهری<|startoftext|>


,generated
0,سهراب سپهریدر گل و من ، ای نسیم سرد و باران ، ای نسیم روشن!ای بهار سبز! در این سبز میدان ، ای بهار شبنم!ای نسیم سرد، ای اشک گرمم ! ای نهال گل ، از این نسیم سوزان ، بگذراز میان گل و شبنم ، به سوی من نگاه کنای نسیم صبح ، ای آب ، ای شبنم! ای شبنم! ای شبنم! ای شبنم! ای شبنم! ای شبنم!من چه دارم ؟ یا چه دارم ؟ ای شبنم!ای نسیم گل! این شبنم ، ای شبنم! ای شبنم! ای شبنم! ای شبنم!
1,سهراب سپهریدر باغ ، و دیوار من ، دیوار من ، دیوار من ، در آب ، و دود چراغ من ، و دود چراغ من ، بر در می شکند .به چشم خودم ، در خواب ، و در رگ های خواب ، و صدا می شکند .در هر جنبشی این رنگ ، و در هر تار ، و در هر پرده ، صدایی می نشیند.می جوشد ، و می ریزد ، و می ریزد ، و می چسبد ، و در وزش زمزمه ، و در هیچ ، و در هر تپش این بوی، گل می بندد ، و می ریزد ، و می
